# Predictive Coding vs Backpropagation: MNIST Reconstruction

Same architecture, same initial weights, same data order, same optimizer. Only the learning algorithm differs.

**Architecture (both models):** `784 → 512 → 256 → 128 → 256 → 512 → 784`, ReLU hidden, Sigmoid output, biases included.

**Backprop autoencoder:** forward pass, loss $\tfrac12\sum_{\text{pixels}}(x-\hat x)^2$, gradients by the chain rule.

**Predictive coding network (PCN):**
- Prediction direction: layer $l-1$ predicts layer $l$: $\mu_l = f_l(W_l x_{l-1} + b_l)$.
- Prediction error: $e_l = x_l - \mu_l$, for $l = 1..6$.
- Energy: $E = \tfrac12\sum_{l=1}^{6}\lVert e_l\rVert^2$.
- Clamping: $x_0$ = image, $x_6$ = image (reconstruction target). $x_1..x_5$ are latent states.
- Inference: $x_l \leftarrow x_l - \eta_x\,\partial E/\partial x_l$ with $\partial E/\partial x_l = e_l - (e_{l+1}\odot f'_{l+1})\,W_{l+1}$, for $T$ steps.
- Learning: $\Delta W_l \propto (e_l \odot f'_l)\,x_{l-1}^\top$, using only quantities local to layer $l$. The resulting gradient is passed to the same Adam optimizer used for backprop.
- Testing: the output is free, so inference settles at the feedforward pass. Test reconstruction is one forward sweep for both models.

**Extra comparisons (beyond the task):**
1. Gradient alignment: cosine similarity between the PCN weight update and the true backprop gradient, per layer, versus number of inference steps $T$.
2. Inference dynamics: how prediction error spreads across layers during PCN inference, compared with backprop's output-only error.
3. Noise robustness: both models reconstruct clean images from noisy inputs.

**How to use:** run the cells top to bottom. Everything (code, experiments, plots, written results) is in this one notebook. The last cell writes `results/RESULTS.md` from the numbers measured in your run.

## 0. Setup

In [ ]:
import sys, time, random
from pathlib import Path
from dataclasses import dataclass
import numpy as np, pandas as pd, torch
import torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RES = ROOT / "results"; RES.mkdir(exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## 1. Configuration (shared by both models)

In [ ]:
@dataclass
class Config:
    # ---- architecture (identical for both models) ----
    dims: tuple = (784, 512, 256, 128, 256, 512, 784)
    hidden_act: str = "relu"      # all hidden layers
    out_act: str = "sigmoid"      # output layer (pixels in [0, 1])

    # ---- shared training settings ----
    batch_size: int = 128
    epochs: int = 10
    lr: float = 1e-3              # weight learning rate (Adam) for BOTH models
    betas: tuple = (0.9, 0.999)
    seed: int = 42
    data_dir: str = "./data"

    # ---- PCN-only inference settings ----
    T: int = 20                   # inference iterations per batch
    lr_x: float = 0.1             # inference (state) learning rate

cfg = Config(data_dir=str(ROOT / 'data'))   # edit epochs / T / lr_x here
cfg

## 2. Models: backprop autoencoder and predictive coding network

In [ ]:
"""
Two models with IDENTICAL structure: 784-512-256-128-256-512-784.

Layer l maps x_{l-1} -> x_l with weights W_l, bias b_l:
    mu_l = f_l(W_l x_{l-1} + b_l),   f_l = ReLU (hidden), Sigmoid (output)

BPAutoencoder : x_0 = image, output mu_6 compared with image, gradients by backprop.
PCN           : x_0 = image (clamped), x_6 = image (clamped target),
                x_1..x_5 are latent states relaxed by energy minimisation;
                weights updated from LOCAL prediction errors only.
Prediction direction in the PCN: layer l-1 predicts layer l (bottom -> top),
    e_l = x_l - mu_l,   E = 1/2 * sum_l ||e_l||^2   (l = 1..6)
"""


class MLP(nn.Module):
    def __init__(self, dims):
        super().__init__()
        self.dims = tuple(dims)
        self.layers = nn.ModuleList(nn.Linear(i, o) for i, o in zip(dims[:-1], dims[1:]))
        self.L = len(self.layers)

    def act(self, i, a):
        return torch.sigmoid(a) if i == self.L - 1 else F.relu(a)

    def dact(self, i, a, mu):
        return mu * (1 - mu) if i == self.L - 1 else (a > 0).to(a.dtype)

    def forward(self, x):
        for i, layer in enumerate(self.layers):
            x = self.act(i, layer(x))
        return x

    def n_params(self):
        return sum(p.numel() for p in self.parameters())


class BPAutoencoder(MLP):
    """Loss = 1/2 * sum_pixels (x - x_hat)^2, averaged over the batch (same scale as PCN energy)."""

    def train_step(self, x, opt):
        opt.zero_grad(set_to_none=True)
        out = self(x)
        loss = 0.5 * ((out - x) ** 2).sum(1).mean()
        loss.backward()
        opt.step()
        return {"mse": F.mse_loss(out.detach(), x)}


class PCN(MLP):
    def __init__(self, dims, T=20, lr_x=0.1):
        super().__init__(dims)
        self.T = T
        self.lr_x = lr_x

    @torch.no_grad()
    def _errors(self, xs):
        es, gs = [None] * (self.L + 1), [None] * (self.L + 1)
        for l in range(1, self.L + 1):
            a = self.layers[l - 1](xs[l - 1])
            mu = self.act(l - 1, a)
            e = xs[l] - mu                      # prediction error at layer l
            es[l] = e
            gs[l] = e * self.dact(l - 1, a, mu)  # error gated by f'(a_l)
        return es, gs

    @torch.no_grad()
    def infer(self, x_in, x_out, T=None, lr_x=None, record=False):
        """Clamp x_0=x_in and x_L=x_out, relax hidden states x_1..x_{L-1} by gradient descent on E.
        Hidden states start from the feedforward pass (all hidden errors = 0 at t=0)."""
        T = self.T if T is None else T
        lr_x = self.lr_x if lr_x is None else lr_x
        xs = [x_in]
        for i, layer in enumerate(self.layers):
            xs.append(self.act(i, layer(xs[-1])))
        init_out = xs[-1]
        xs[-1] = x_out.clone()
        trace = []
        for _ in range(T):
            es, gs = self._errors(xs)
            if record:
                trace.append(torch.stack([0.5 * e.pow(2).sum(1).mean() for e in es[1:]]))
            for l in range(1, self.L):
                # dE/dx_l = e_l - (e_{l+1} * f') W_{l+1}
                xs[l] = xs[l] - lr_x * (es[l] - gs[l + 1] @ self.layers[l].weight)
        es, gs = self._errors(xs)
        if record:
            trace.append(torch.stack([0.5 * e.pow(2).sum(1).mean() for e in es[1:]]))
        energy = sum(0.5 * e.pow(2).sum(1).mean() for e in es[1:])
        return {"xs": xs, "es": es, "gs": gs, "init_out": init_out, "energy": energy,
                "trace": torch.stack(trace) if record else None}

    @torch.no_grad()
    def weight_grads(self, out):
        """dE/dW_l and dE/db_l from local quantities: -(e_l*f') x_{l-1}^T."""
        xs, gs = out["xs"], out["gs"]
        B = xs[0].size(0)
        gw = [-(gs[l].T @ xs[l - 1]) / B for l in range(1, self.L + 1)]
        gb = [-gs[l].mean(0) for l in range(1, self.L + 1)]
        return gw, gb

    @torch.no_grad()
    def train_step(self, x, opt):
        out = self.infer(x, x)
        gw, gb = self.weight_grads(out)
        for layer, w, b in zip(self.layers, gw, gb):
            layer.weight.grad, layer.bias.grad = w, b
        opt.step()
        opt.zero_grad(set_to_none=True)
        return {"mse": F.mse_loss(out["init_out"], x), "energy": out["energy"]}

## 3. Data, training loop, evaluation and analysis utilities

In [ ]:
def get_loaders(cfg):
    """MNIST, pixels in [0,1]. Same seed => same batch order for both models."""
    tf = transforms.ToTensor()
    train = datasets.MNIST(cfg.data_dir, train=True, download=True, transform=tf)
    test = datasets.MNIST(cfg.data_dir, train=False, download=True, transform=tf)
    g = torch.Generator().manual_seed(cfg.seed)
    train_loader = DataLoader(train, batch_size=cfg.batch_size, shuffle=True, generator=g)
    test_loader = DataLoader(test, batch_size=1000, shuffle=False)
    return train_loader, test_loader

In [ ]:
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


def _sync(device):
    if device.type == "cuda":
        torch.cuda.synchronize()


@torch.no_grad()
def evaluate(model, loader, device):
    """Per-pixel reconstruction MSE (single feedforward pass for both models)."""
    model.eval()
    tot, n = 0.0, 0
    for x, _ in loader:
        x = x.to(device).flatten(1)
        tot += ((model(x) - x) ** 2).sum().item()
        n += x.numel()
    return tot / n


def fit(model, train_loader, test_loader, cfg, device, name=""):
    """Identical loop for both models; only model.train_step differs."""
    opt = torch.optim.Adam(model.parameters(), lr=cfg.lr, betas=cfg.betas)
    h = {"train_mse": [], "test_mse": [evaluate(model, test_loader, device)],
         "epoch_time": [], "energy": []}
    for ep in range(1, cfg.epochs + 1):
        model.train()
        _sync(device); t0 = time.perf_counter()
        tr, en, n = torch.zeros((), device=device), torch.zeros((), device=device), 0
        for x, _ in train_loader:
            x = x.to(device).flatten(1)
            s = model.train_step(x, opt)
            tr += s["mse"] * x.size(0)
            if "energy" in s:
                en += s["energy"] * x.size(0)
            n += x.size(0)
        _sync(device); dt = time.perf_counter() - t0
        h["train_mse"].append((tr / n).item())
        h["energy"].append((en / n).item())
        h["epoch_time"].append(dt)
        h["test_mse"].append(evaluate(model, test_loader, device))
        print(f"[{name}] epoch {ep:2d}/{cfg.epochs} | train MSE {h['train_mse'][-1]:.5f} | "
              f"test MSE {h['test_mse'][-1]:.5f} | {dt:.1f}s")
    return h


@torch.no_grad()
def time_inference(model, loader, device, repeats=3):
    """Seconds to reconstruct the whole test set (best of `repeats`)."""
    model.eval()
    best = float("inf")
    for _ in range(repeats):
        _sync(device); t0 = time.perf_counter()
        for x, _ in loader:
            model(x.to(device).flatten(1))
        _sync(device)
        best = min(best, time.perf_counter() - t0)
    return best

In [ ]:
"""Extra comparisons that expose how the two learning rules differ."""


def bp_weight_grads(model, x):
    """Backprop gradient of 1/2*sum_pixels(x - x_hat)^2 (batch mean) w.r.t. each weight matrix."""
    out = model(x)
    loss = 0.5 * ((out - x) ** 2).sum(1).mean()
    return [g.detach() for g in torch.autograd.grad(loss, [l.weight for l in model.layers])]


def _cos(a, b):
    return F.cosine_similarity(a.flatten(), b.flatten(), dim=0).item()


def alignment_vs_T(pcn, x, Ts):
    """Cosine similarity between PCN weight update and BP gradient, per layer, per #inference steps.
    Returns array [len(Ts), L]."""
    gb = bp_weight_grads(pcn, x)
    rows = []
    for T in Ts:
        gw, _ = pcn.weight_grads(pcn.infer(x, x, T=T))
        rows.append([_cos(a, b) for a, b in zip(gw, gb)])
    return np.array(rows)


def per_layer_energy_trace(pcn, x, T=None):
    """[T+1, L] layer-wise energy 1/2||e_l||^2 during inference (t=0 is the feedforward state)."""
    return pcn.infer(x, x, T=T, record=True)["trace"].cpu().numpy()


@torch.no_grad()
def noise_robustness(models, loader, device, sigmas, seed=0):
    """Test-time Gaussian input noise; reconstruct and compare with the CLEAN image."""
    res = {k: [] for k in models}
    for s in sigmas:
        g = torch.Generator(device="cpu").manual_seed(seed)
        tot = {k: 0.0 for k in models}; n = 0
        for x, _ in loader:
            x = x.flatten(1)
            xn = (x + s * torch.randn(x.shape, generator=g)).clamp(0, 1)
            x, xn = x.to(device), xn.to(device)
            for k, m in models.items():
                m.eval()
                tot[k] += ((m(xn) - x) ** 2).sum().item()
            n += x.numel()
        for k in models:
            res[k].append(tot[k] / n)
    return res

## 4. Data

In [ ]:
set_seed(cfg.seed)
train_loader, test_loader = get_loaders(cfg)
print(len(train_loader.dataset), "train |", len(test_loader.dataset), "test | batch", cfg.batch_size)
xb, yb = next(iter(test_loader))
fig, ax = plt.subplots(1, 10, figsize=(14, 1.8))
for i in range(10):
    ax[i].imshow(xb[i, 0], cmap="gray"); ax[i].set_title(int(yb[i])); ax[i].axis("off")
plt.suptitle("MNIST test samples"); plt.show()

## 5. Models: identical architecture and identical initial weights

In [ ]:
set_seed(cfg.seed)
bp = BPAutoencoder(cfg.dims).to(device)
pcn = PCN(cfg.dims, T=cfg.T, lr_x=cfg.lr_x).to(device)
pcn.load_state_dict(bp.state_dict())          # same init

shapes = [tuple(l.weight.shape) for l in bp.layers]
assert shapes == [tuple(l.weight.shape) for l in pcn.layers] and bp.n_params() == pcn.n_params()
print(pd.DataFrame({"layer": range(1, len(shapes)+1), "W shape (out,in)": shapes,
                    "activation": ["ReLU"]*(len(shapes)-1) + ["Sigmoid"]}).to_string(index=False))
print(f"\ntrainable parameters  BP: {bp.n_params():,} | PCN: {pcn.n_params():,}")
print(f"optimizer: Adam(lr={cfg.lr}, betas={cfg.betas}) | batch {cfg.batch_size} | epochs {cfg.epochs} | seed {cfg.seed}")
print(f"PCN inference: T={cfg.T}, lr_x={cfg.lr_x}, hidden states initialised by a forward pass")

## 6. Train the backpropagation autoencoder

In [ ]:
train_loader, test_loader = get_loaders(cfg)      # re-create so batch order is seeded identically
hist_bp = fit(bp, train_loader, test_loader, cfg, device, name="BP ")

## 7. Train the predictive coding network

In [ ]:
train_loader, test_loader = get_loaders(cfg)      # same seed -> same batch order as BP
hist_pc = fit(pcn, train_loader, test_loader, cfg, device, name="PCN")

## 8. Convergence: reconstruction MSE over epochs

In [ ]:
ep = np.arange(0, cfg.epochs + 1)
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].plot(ep[1:], hist_bp["train_mse"], "o-", label="BP"); ax[0].plot(ep[1:], hist_pc["train_mse"], "s-", label="PCN")
ax[0].set_title("Train MSE (per pixel)"); ax[0].set_xlabel("epoch"); ax[0].set_yscale("log"); ax[0].legend()
ax[1].plot(ep, hist_bp["test_mse"], "o-", label="BP"); ax[1].plot(ep, hist_pc["test_mse"], "s-", label="PCN")
ax[1].set_title("Test MSE (epoch 0 = untrained)"); ax[1].set_xlabel("epoch"); ax[1].set_yscale("log"); ax[1].legend()
ax[2].plot(ep[1:], hist_pc["energy"], "s-", color="C1"); ax[2].set_title("PCN energy E after inference (train)")
ax[2].set_xlabel("epoch"); ax[2].set_yscale("log")
plt.tight_layout(); plt.savefig(RES / "convergence.png", dpi=150); plt.show()

## 9. Sample reconstructions

In [ ]:
x = xb[:10].to(device).flatten(1)
with torch.no_grad():
    r_bp, r_pc = bp(x).cpu(), pcn(x).cpu()
x = x.cpu()
rows = [("Original", x), ("BP", r_bp), ("PCN", r_pc), ("|BP - orig|", (r_bp - x).abs()), ("|PCN - orig|", (r_pc - x).abs())]
fig, ax = plt.subplots(len(rows), 10, figsize=(14, 7.2))
for r, (name, imgs) in enumerate(rows):
    for c in range(10):
        ax[r, c].imshow(imgs[c].view(28, 28), cmap="gray" if r < 3 else "magma", vmin=0, vmax=1)
        ax[r, c].axis("off")
    ax[r, 0].text(-0.1, 0.5, name, transform=ax[r, 0].transAxes, ha="right", va="center", fontsize=10)
plt.tight_layout(); plt.savefig(RES / "reconstructions.png", dpi=150); plt.show()

## 10. Quantitative comparison: error and computational cost

In [ ]:
t_inf_bp = time_inference(bp, test_loader, device)
t_inf_pc = time_inference(pcn, test_loader, device)
summary = pd.DataFrame({
    "Final train MSE": [hist_bp["train_mse"][-1], hist_pc["train_mse"][-1]],
    "Final test MSE":  [hist_bp["test_mse"][-1],  hist_pc["test_mse"][-1]],
    "Train time / epoch (s)": [np.mean(hist_bp["epoch_time"]), np.mean(hist_pc["epoch_time"])],
    "Total train time (s)": [np.sum(hist_bp["epoch_time"]), np.sum(hist_pc["epoch_time"])],
    "Test-set inference time (s)": [t_inf_bp, t_inf_pc],
    "Params": [bp.n_params(), pcn.n_params()],
}, index=["Backprop AE", "Predictive Coding"])
summary.to_csv(RES / "summary.csv")
display(summary.style.format({"Final train MSE": "{:.5f}", "Final test MSE": "{:.5f}", "Train time / epoch (s)": "{:.2f}",
                              "Total train time (s)": "{:.1f}", "Test-set inference time (s)": "{:.3f}", "Params": "{:,}"}))
print(f"PCN / BP training-time ratio: {summary.iloc[1,3] / summary.iloc[0,3]:.1f}x")

## 11. Extra comparison 1: does the local PC update approximate the backprop gradient?

Both rules start from the same weights and the same batch. For each layer, we compute the cosine similarity between the PCN weight update (from $T$ inference steps) and the exact backprop gradient of the same loss. A value of 1 means the same update direction.

In [ ]:
xa = next(iter(test_loader))[0][:512].to(device).flatten(1)
Ts = [1, 2, 3, 4, 5, 6, 8, 10, 15, 20, 30, 50, 100, 200]

fresh = PCN(cfg.dims).to(device); set_seed(cfg.seed); fresh.load_state_dict(BPAutoencoder(cfg.dims).state_dict())
A_init = alignment_vs_T(fresh, xa, Ts)       # untrained weights
A_trn  = alignment_vs_T(pcn, xa, Ts)         # trained PCN weights

fig, ax = plt.subplots(1, 2, figsize=(14, 4.2), sharey=True)
for a, A, title in [(ax[0], A_init, "Untrained weights"), (ax[1], A_trn, "Trained PCN weights")]:
    for l in range(A.shape[1]):
        a.plot(Ts, A[:, l], "o-", ms=3, label=f"W{l+1}" + (" (input side)" if l == 0 else " (output side)" if l == A.shape[1]-1 else ""))
    a.axvline(cfg.T, color="gray", ls="--", lw=1); a.set_xscale("log"); a.set_title(title)
    a.set_xlabel("inference steps T (dashed = training T)"); a.set_ylim(-0.1, 1.05)
ax[0].set_ylabel("cosine(PC update, BP gradient)"); ax[1].legend(fontsize=8, loc="lower left")
plt.tight_layout(); plt.savefig(RES / "gradient_alignment.png", dpi=150); plt.show()
print("Cosine at training T =", cfg.T, "(trained weights):", np.round(A_trn[Ts.index(cfg.T)], 3))

## 12. Extra comparison 2: where does the error live during inference?

Backprop computes the error only at the output and sends it backwards in a separate pass. In a PCN the error is a quantity at every layer. It starts at the output layer when the target is clamped and spreads backwards through the hierarchy as the latent states relax.

In [ ]:
trace = per_layer_energy_trace(pcn, xa, T=40)    # [T+1, 6]
fig, ax = plt.subplots(1, 2, figsize=(14, 4.2))
for l in range(trace.shape[1]):
    ax[0].plot(trace[:, l], label=f"layer {l+1}  (e{l+1})")
ax[0].plot(trace.sum(1), "k--", lw=2, label="total E"); ax[0].set_yscale("log")
ax[0].set_xlabel("inference step t"); ax[0].set_ylabel("1/2 ||e_l||²  (batch mean)"); ax[0].set_title("Layer-wise prediction error during PCN inference")
ax[0].axvline(cfg.T, color="gray", ls=":"); ax[0].legend(fontsize=8)
snap = [0, 5, 10, cfg.T, 40]
for t in snap:
    ax[1].plot(range(1, trace.shape[1] + 1), trace[t] / trace[t].sum(), "o-", label=f"t = {t}")
ax[1].set_xlabel("layer l"); ax[1].set_ylabel("share of total energy"); ax[1].set_title("Error distribution across layers (BP: 100% at layer 6)")
ax[1].legend()
plt.tight_layout(); plt.savefig(RES / "inference_dynamics.png", dpi=150); plt.show()

## 13. Extra comparison 3: robustness to input noise at test time

In [ ]:
sigmas = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]
rob = noise_robustness({"BP": bp, "PCN": pcn}, test_loader, device, sigmas)
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].plot(sigmas, rob["BP"], "o-", label="BP"); ax[0].plot(sigmas, rob["PCN"], "s-", label="PCN")
ax[0].set_xlabel("noise std σ"); ax[0].set_ylabel("MSE vs clean image"); ax[0].set_title("Denoising-by-reconstruction MSE"); ax[0].legend()

g = torch.Generator().manual_seed(1)
xc = xb[:6].flatten(1); xn = (xc + 0.4 * torch.randn(xc.shape, generator=g)).clamp(0, 1)
with torch.no_grad():
    o_bp, o_pc = bp(xn.to(device)).cpu(), pcn(xn.to(device)).cpu()
grid = torch.cat([torch.cat(list(m.view(-1, 28, 28)), dim=1) for m in (xc, xn, o_bp, o_pc)], dim=0)
ax[1].imshow(grid, cmap="gray"); ax[1].axis("off"); ax[1].set_title("rows: clean | noisy (σ=0.4) | BP | PCN")
plt.tight_layout(); plt.savefig(RES / "noise_robustness.png", dpi=150); plt.show()
pd.DataFrame(rob, index=[f"σ={s}" for s in sigmas]).style.format("{:.5f}")

## 14. Results description (auto-generated from this run)

This cell builds the written results from the measured numbers and saves them to `results/RESULTS.md`.

In [ ]:
b, p = summary.loc["Backprop AE"], summary.loc["Predictive Coding"]
it = Ts.index(cfg.T)
tr_ratio = p["Total train time (s)"] / b["Total train time (s)"]
bp_final, pc_final = b["Final test MSE"], p["Final test MSE"]
lower = "backpropagation" if bp_final < pc_final else "predictive coding"
gap = abs(bp_final - pc_final) / max(min(bp_final, pc_final), 1e-12) * 100
reach = next((i for i, v in enumerate(hist_pc["test_mse"]) if v <= bp_final), None)
bp_half = next((i for i, v in enumerate(hist_bp["test_mse"]) if v <= 2 * bp_final), None)
e0, eT = trace[0] / trace[0].sum(), trace[min(cfg.T, len(trace)-1)] / trace[min(cfg.T, len(trace)-1)].sum()
noise_i = sigmas.index(0.3)
rob_better = "backprop" if rob["BP"][noise_i] < rob["PCN"][noise_i] else "PCN"
inf_ratio = p["Test-set inference time (s)"] / b["Test-set inference time (s)"]

txt = f"""# Results: Predictive Coding vs Backpropagation (MNIST reconstruction)

Generated by `notebooks/PC_vs_Backprop.ipynb` on device `{device}`.

## Setup
- Architecture (both): {" -> ".join(map(str, cfg.dims))}, ReLU hidden, Sigmoid output, {bp.n_params():,} parameters each.
- Shared: batch size {cfg.batch_size}, {cfg.epochs} epochs, Adam (lr {cfg.lr}), seed {cfg.seed}, identical initial weights and batch order.
- PCN inference: T = {cfg.T} steps, state learning rate {cfg.lr_x}; both input and output clamped to the image during training.

## Main results
| Model | Final train MSE | Final test MSE | Train time / epoch (s) | Total train time (s) | Test-set inference (s) |
|---|---|---|---|---|---|
| Backprop AE | {b["Final train MSE"]:.5f} | {bp_final:.5f} | {b["Train time / epoch (s)"]:.2f} | {b["Total train time (s)"]:.1f} | {b["Test-set inference time (s)"]:.3f} |
| Predictive Coding | {p["Final train MSE"]:.5f} | {pc_final:.5f} | {p["Train time / epoch (s)"]:.2f} | {p["Total train time (s)"]:.1f} | {p["Test-set inference time (s)"]:.3f} |

**Reconstruction quality.** {lower.capitalize()} reached the lower final test MSE, by {gap:.1f}% relative to the better model.

**Convergence.** Test MSE per epoch (epoch 0 = untrained): BP {", ".join(f"{v:.4f}" for v in hist_bp["test_mse"])}; PCN {", ".join(f"{v:.4f}" for v in hist_pc["test_mse"])}. """ + (f"The PCN matched BP's final test MSE at epoch {reach}." if reach is not None else "The PCN did not reach BP's final test MSE within the epochs run.") + f"""

**Computational cost.** PCN training took {tr_ratio:.1f}x the time of backprop training, because each batch needs {cfg.T} inference sweeps before the weight update. Test-set inference time was {inf_ratio:.2f}x BP's, since both use one forward sweep at test time.

## Extra comparison 1: PC update vs backprop gradient
Cosine similarity per layer (W1 input side ... W6 output side) at T = {cfg.T}, trained weights: {np.round(A_trn[it], 3).tolist()}; untrained weights: {np.round(A_init[it], 3).tolist()}. Values near 1 mean the local PC update points in nearly the same direction as the backprop gradient. Figure: `results/gradient_alignment.png`.

## Extra comparison 2: where the error lives during inference
Share of total energy per layer (layers 1..6) at t = 0: {np.round(e0, 3).tolist()}; at t = {cfg.T}: {np.round(eT, 3).tolist()}. Backprop has all of its error at the output layer; the PCN spreads it across layers as inference proceeds. Figure: `results/inference_dynamics.png`.

## Extra comparison 3: noise robustness
Test MSE against the clean image, noisy input (sigma = 0.3): BP {rob["BP"][noise_i]:.5f}, PCN {rob["PCN"][noise_i]:.5f} ({rob_better} lower). Full curve for sigma 0 to 0.5 in `results/noise_robustness.png`. Both models are trained for plain reconstruction, with no noise during training.

## Discussion
- **Fairness.** Architecture, initial weights, optimizer, data order and epochs are identical, so differences come from how weight gradients are obtained.
- **Advantages of PC.** Weight updates are local (own-layer error times the activity below), with no backward pass or weight transport. Errors are explicit at every layer, and the same energy framework supports bidirectional inference.
- **Limitations of PC.** Training costs about {tr_ratio:.1f}x more here because of iterative inference, and T and the state learning rate need tuning. With longer inference, the output-layer update drifts from the backprop gradient (see extra comparison 1).
- **Caveat.** Results come from a single seed and {cfg.epochs} epochs. Repeating with several seeds would show whether small differences are meaningful.

## Figures
`results/convergence.png`, `results/reconstructions.png`, `results/gradient_alignment.png`, `results/inference_dynamics.png`, `results/noise_robustness.png`.
"""
(RES / "RESULTS.md").write_text(txt)
from IPython.display import Markdown, display
display(Markdown(txt))